# FABLE Pyculator 2020 Notebook Loop

This notebook demonstrates the first FABLE Pyculator loop for the public 2020 FABLE-C workbook and an ignored local Modelwright-generated Python model.

Use the VSCode notebook kernel from the `.venv` created in the `fable-pyculator` repository root. The setup cell prints the active Python executable and warns if the selected kernel does not appear to be that repo-local environment.

Expected local artifacts, resolved relative to the `fable-pyculator` repository root:

- `tmp/private-workbooks/2020_Open_FABLECalculator.xlsx`
- `tmp/generated-models/fable-2020/generated_fable_2020_model.py`

If the generated model is missing but a sibling Modelwright checkout exists, this notebook can materialize the local compressed Modelwright example into the ignored FABLE Pyculator `tmp/` path. If required artifacts are still unavailable, the notebook reports that status and skips execution cells instead of failing during setup.


In [ ]:
from pathlib import Path
import sys
import lzma
import shutil

from IPython.display import Markdown, display

from fable_pyculator import (
    DEFAULT_2020_GENERATED_MODEL_PATH,
    DEFAULT_2020_WORKBOOK_PATH,
    build_2020_notebook_spec,
    load_generated_model,
    run_notebook_loop,
)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    candidates.extend(parent / "fable-pyculator" for parent in [start, *start.parents])
    for candidate in candidates:
        pyproject = candidate / "pyproject.toml"
        if pyproject.exists() and "name = \"fable-pyculator\"" in pyproject.read_text(encoding="utf-8"):
            return candidate
    raise RuntimeError("Could not find the fable-pyculator repository root.")


notebook_working_dir = Path.cwd().resolve()
repo_root = find_repo_root(notebook_working_dir)
expected_kernel_prefix = repo_root / ".venv"
active_python = Path(sys.executable)
active_prefix = Path(sys.prefix).resolve()
using_repo_venv = active_prefix == expected_kernel_prefix
workbook_path = repo_root / DEFAULT_2020_WORKBOOK_PATH
generated_model_path = repo_root / DEFAULT_2020_GENERATED_MODEL_PATH
modelwright_archive = repo_root.parent / "modelwright" / "examples" / "fable_2020" / "generated_fable_2020_model.py.xz"

if not generated_model_path.exists() and modelwright_archive.exists():
    generated_model_path.parent.mkdir(parents=True, exist_ok=True)
    with lzma.open(modelwright_archive, "rb") as source, generated_model_path.open("wb") as target:
        shutil.copyfileobj(source, target)

missing = [path for path in (workbook_path, generated_model_path) if not path.exists()]
ARTIFACTS_READY = not missing

if ARTIFACTS_READY:
    display(Markdown(
        f"Notebook working directory: `{notebook_working_dir}`\n\n"
        f"Resolved fable-pyculator repo root: `{repo_root}`\n\n"
        f"Active Python: `{active_python}`\n\n"
        f"Active environment prefix: `{active_prefix}`\n\n"
        f"Expected VSCode kernel prefix: `{expected_kernel_prefix}`\n\n"
        + ("" if using_repo_venv else "**Warning:** select the repo-local `.venv` kernel in VSCode before running model cells.\n\n")
        + f"Workbook: `{workbook_path}`\n\n"
        f"Generated model: `{generated_model_path}`"
    ))
else:
    display(Markdown(
        f"Notebook working directory: `{notebook_working_dir}`\n\n"
        f"Resolved fable-pyculator repo root: `{repo_root}`\n\n"
        f"Active Python: `{active_python}`\n\n"
        f"Active environment prefix: `{active_prefix}`\n\n"
        f"Expected VSCode kernel prefix: `{expected_kernel_prefix}`\n\n"
        + ("" if using_repo_venv else "**Warning:** select the repo-local `.venv` kernel in VSCode before running model cells.\n\n")
        + "Required local artifacts are missing, so execution cells will be skipped.\n\n"
        + "\n".join(f"- `{path}`" for path in missing)
        + "\n\nRestore the workbook under `tmp/private-workbooks/` and the generated model under "
        + "`tmp/generated-models/fable-2020/`, then rerun the notebook."
    ))


## Build the Workbook-Derived Spec

The spec discovers the 2020 scenario selection controls, canonical output tables, and curated FOOD/LAND/GHG/WATER headline series.

In [ ]:
if ARTIFACTS_READY:
    spec = build_2020_notebook_spec(workbook_path)
    display((len(spec.selection_controls), len(spec.output_tables), len(spec.headline_series)))
else:
    spec = None
    display(Markdown("Skipped: local workbook/model artifacts are not ready."))


## Choose Scenario Selections

Selection values are named by discovered table names such as `gdp_scen`. Each value expands to the workbook marker cells that clear all rows except the chosen option.

In [ ]:
selections = {
    "gdp_scen": "SSP1",
}

if ARTIFACTS_READY:
    spec.input_mapping(selections)
else:
    selections


## Run the Generated Model

The generated model is imported from ignored `tmp/` space. The wrapper returns raw run metadata, selected output tables, headline DataFrames, and optional matplotlib figures.

In [ ]:
if ARTIFACTS_READY:
    generated_model = load_generated_model(generated_model_path)
    result = run_notebook_loop(
        generated_model,
        spec,
        selections,
        scenario_name="ssp1_demo",
        output_table_names=("ghg_resultsghg",),
        headline_series_names=("food_total_kcal_feas", "land_total_area", "ghg_total_co2e", "water_total_footprint"),
    )
    display((result.run.scenario_name, len(result.run.inputs), len(result.run.values)))
else:
    result = None
    display(Markdown("Skipped: local workbook/model artifacts are not ready."))


## Inspect Outputs

In [ ]:
if result is not None:
    result.output_tables["ghg_resultsghg"].head()


In [ ]:
if result is not None:
    result.headline_frames["ghg_total_co2e"]


In [ ]:
if result is not None:
    result.headline_figures["ghg_total_co2e"]
